In [0]:
%sql
USE CATALOG databricksbootcamp;
CREATE SCHEMA IF NOT EXISTS santeflux;

In [0]:
%sql
USE SCHEMA santeflux;
CREATE VOLUME IF NOT EXISTS santefluxvol

In [0]:
BASE = "/Volumes/databricksbootcamp/santeflux/santefluxvol"
events = spark.read.csv(f"{BASE}/events.csv", header=True, inferSchema=True)
patients = spark.read.csv(f"{BASE}/patients.csv", header=True, inferSchema=True)

display(events)
display(patients)

In [0]:
dbutils.widgets.text("region", "Paris")
region = dbutils.widgets.get("region")

In [0]:
%run "/Workspace/Users/nargideharsh@gmail.com/Sante_Flux_Case_Study/SanteFlux_Functions"

In [0]:
from pyspark.sql.functions import *

In [0]:
joined_df = patients.join(events, "patient_id", "left").drop(events.region)

In [0]:
display(joined_df)

In [0]:
enriched_join = joined_df.withColumn("Age_Bracket", age_band(col("date_of_birth"))).withColumn("Cost_Flag", cost_flag(col("cost_eur"))).withColumn("Risk_Score", risk_score(col("date_of_birth"), col("diagnosis_code"), col("cost_eur")))


In [0]:
display(enriched_join)

In [0]:
unq_region = enriched_join.select("region").distinct().collect()

In [0]:
print(unq_region)

In [0]:
reg = []
for r in unq_region:
    reg.append(r[0])
print(reg)

In [0]:
for re in reg:
    cnt = events.filter(col("region") == re).count()
    print(f"{re}: {cnt}")

In [0]:
enriched_join.write.mode("overwrite").saveAsTable("databricksbootcamp.santeflux.patient_risk")

In [0]:
patient_risk_tbl = spark.read.table("databricksbootcamp.santeflux.patient_risk")

In [0]:
display(patient_risk_tbl)

In [0]:
%sql
UPDATE databricksbootcamp.santeflux.patient_risk
SET Risk_Score = 0

In [0]:
%sql
DESCRIBE HISTORY databricksbootcamp.santeflux.patient_risk

In [0]:
%sql
SELECT * FROM databricksbootcamp.santeflux.patient_risk VERSION AS OF 1

In [0]:
%sql
RESTORE TABLE databricksbootcamp.santeflux.patient_risk VERSION AS OF 0